In [1]:
import gymnasium as gym
from stable_baselines3 import SAC
from stable_baselines3.common.env_util import make_vec_env
import torch

In [2]:
if torch.cuda.is_available():
    dev = "cuda:0"
elif torch.backends.mps.is_available():
    dev = "mps"
else:
    dev = "cpu"
device = torch.device(dev)
device

device(type='mps')

In [3]:
device = torch.device("cpu")
device

device(type='cpu')

SAC does NOT work natively with CartPole, because:

* SAC requires continuous action spaces
* CartPole uses discrete actions (left/right)

So we only use Pendulum, which is a continuous env:

In [4]:
env = env = make_vec_env("Pendulum-v1", n_envs=4)

In [5]:
model = SAC("MlpPolicy", env, verbose=1, device=device)
model.learn(total_timesteps=200_000)

Using cpu device
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 200       |
|    ep_rew_mean     | -1.15e+03 |
| time/              |           |
|    episodes        | 4         |
|    fps             | 1584      |
|    time_elapsed    | 0         |
|    total_timesteps | 800       |
| train/             |           |
|    actor_loss      | 9.22      |
|    critic_loss     | 1.63      |
|    ent_coef        | 0.949     |
|    ent_coef_loss   | -0.0888   |
|    learning_rate   | 0.0003    |
|    n_updates       | 174       |
----------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 200      |
|    ep_rew_mean     | -1.2e+03 |
| time/              |          |
|    episodes        | 8        |
|    fps             | 1495     |
|    time_elapsed    | 1        |
|    total_timesteps | 1600     |
| train/             |          |
|    actor_loss      | 12.7     |
|    critic_lo

In [6]:
test_env = gym.make("Pendulum-v1")

num_eval_episodes = 100
episode_rewards = []

for episode in range(num_eval_episodes):
    state, _ = test_env.reset()
    done = False
    total_reward = 0.0

    while not done:
        action, _ = model.predict(state, deterministic=True)
        state, reward, terminated, truncated, _ = test_env.step(action)
        done = terminated or truncated

        total_reward += reward

    episode_rewards.append(total_reward)

test_env.close()

average_reward = sum(episode_rewards) / num_eval_episodes
print(f"Average reward over {num_eval_episodes} episodes: {average_reward:.2f}")

Average reward over 100 episodes: -132.91
